# Lab 00: LLM Basics

This notebook is the on-ramp for the lab. It assumes no prior knowledge about language models. By the end you will understand every line of the API call that the rest of the course is built on.

What you will work through:

1. **What is a large language model?** A program that continues text, nothing else.
2. **Tokens**: how the model reads text, and why it sometimes fails at tasks a child finds easy.
3. **Where the model runs**: the GWDG Academic Cloud, and how your notebook reaches it.
4. **The chat call**: every argument of `llm.chat()`, one at a time.
5. **Same prompt, different answers**: why answers vary between runs, and how to control that.

This notebook is **not graded**. Run every cell, change things, break things. The labs that follow, starting with Lab 01, assume you have seen what is in here.

Nothing to install: the model runs on the GWDG Academic Cloud and is reached through our small client `llm.py`. Do the Onboarding notebook first to save your API key.

## Learning goals

By the end of this notebook you should be able to:

- say in one sentence what a large language model does
- explain what a token is and name one consequence of tokenization
- say where the model runs and what our `llm` Python module does
- call `llm.chat()` and explain every argument and the main response fields
- explain why the same prompt can give different answers, and what temperature changes

## Setup check

Run the cell below. If it prints a short reply, your API key and the connection to the model work. If it errors, run the Onboarding notebook first; if it still errors, ask for help before continuing.

You are not expected to understand this cell yet.

In [ ]:
import llm

MODEL = llm.CHAT_MODEL   # the model every cell uses; Part 3 explains its name

# Your first LLM call: send one message, print the reply.
response = llm.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "In one sentence, what is a large language model?"}],
)
print(response["choices"][0]["message"]["content"])

## Part 1: What is a large language model?

A **large language model** (LLM) is a program that takes text and predicts a likely continuation of that text. That is the entire interface: text in, text out.

```
 input text                                  likely continuation
 ──────────                                  ───────────────────
 "The capital of France is"   →  [ LLM ]  →  " Paris."
```

The model learned this by reading enormous amounts of text during training and adjusting itself until its continuations matched what the text actually said. Four consequences hold for the whole course:

- **It continues text.** A "chat" is text too: the conversation so far goes in, a plausible next reply comes out.
- **It has no memory.** Each call starts from zero. What looks like memory in chat apps is resent conversation history (later labs show this in detail).
- **Its knowledge is frozen.** Training ended on some date (the **knowledge cutoff**), and the model sees nothing outside the conversation: no internet, no clock, no files. When ChatGPT tells you today's weather, the model is not looking anything up; a **tool** fetches the weather and pastes it into the prompt. Building such tools is where this course is going.
- **It produces plausible text, not verified text.** Nothing in the mechanism checks facts.

<details><summary>Where do the predictions come from?</summary>

The model is a neural network: billions of adjustable numbers (called **parameters** or **weights**). During training, the network reads text from books, websites, and code, and the numbers are tuned so that its predicted continuations match the real ones. Afterwards it is fine-tuned on example dialogues so that "continue the text" behaves like "answer the question". This principle is universal: ChatGPT, Claude, Gemini, and the model in this lab all work this way.

</details>

### Reading the code cells

Every code cell in this notebook makes the same call:

```python
response = llm.chat(
    model=MODEL,                                    # which model to use
    messages=[{"role": "user", "content": "..."}],  # your message to it
)
print(response["choices"][0]["message"]["content"])  # the model's reply text
```

`MODEL` was set in the setup cell to `llm.CHAT_MODEL`, the everyday model of this course. `messages` is a Python list of dictionaries. Here it holds one dictionary with two keys: `"role"` says who is speaking (`"user"` is you), `"content"` is the text. The reply comes back in `response["choices"][0]["message"]["content"]`: `choices` is a list because a server can return several alternative replies, and we always ask for one, so we take the first (`[0]`).

That is all you need to read the cells that follow. Why `messages` is a list, what other roles exist, and what else the response contains: Part 4 takes the call apart piece by piece.

In [ ]:
# Through the chat interface we have to *ask* for a continuation.
response = llm.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Continue this sentence, then stop: 'The three most common materials in mechanical engineering are'"}],
)
print(response["choices"][0]["message"]["content"])

### Plausible is not the same as true

When the model produces fluent text that is factually wrong, this is called a **hallucination**. It is not a rare glitch. It is the default failure mode of a system that predicts likely text: a made-up citation in the right format is, statistically, a very plausible continuation.

In [ ]:
# This paper does not exist. Watch what the model does anyway.
response = llm.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Give the exact title, authors, and journal of the 2018 paper that first reported fatigue testing of 3D-printed titanium bicycle cranksets."}],
)
print(response["choices"][0]["message"]["content"])

Two outcomes are possible, and both are informative:

- The model **invents** a plausible-sounding title, authors, and journal. Nothing in it is real, but it is formatted exactly like a real citation.
- The model **admits** it does not know. Newer models are trained to do this more often, because the underlying mechanism does not do it by itself.

Your output will also differ from your neighbor's, and from your own next run. Part 5 explains why.

The working rule for this course: **an LLM answer is a draft, not a source.** Verify anything that matters.

### Try it

- Ask a question from your own field where you can verify the answer yourself.
- Ask about something the model cannot possibly know (today's weather, this week's news) and watch how it handles not knowing.

In [ ]:
# Scratchpad: copy a cell from above, modify, run.


## Part 2: Tokens: how the model reads text

The model never sees letters or words. Before your text reaches it, a **tokenizer** splits the text into **tokens**: chunks of a few characters that form the model's vocabulary.

```
"the bridge is closed"
    →  ["the", " bridge", " is", " closed"]          4 tokens

"Geschwindigkeitsbegrenzung"
    →  ["G", "eschwind", "igkeit", "s", "begrenzung"]  5 tokens (illustrative)
```

Common English words are usually one token each. Rare words, German compounds, and technical terms shatter into several pieces. The exact split is **model-specific**: every model family has its own tokenizer, fixed before the model is trained.

One consequence: the model cannot look inside a token. Questions about individual letters, which any child can answer by looking, are surprisingly hard for it.

In [ ]:
# The model has to answer a question about letters it cannot directly see.
word = "Geschwindigkeitsbegrenzung"

response = llm.chat(
    model=MODEL,
    messages=[{"role": "user", "content": f"How many times does the letter 'e' appear in the word '{word}'? Answer with just the number."}],
)
print("model says:", response["choices"][0]["message"]["content"])
print("actually:  ", word.count("e"))

If your model got it right: good, but do not be fooled. Newer models are trained to spell words out letter by letter in their internal reasoning before answering, precisely because they cannot see letters directly. The workaround exists because the limitation is real. Try a longer or stranger word in the scratchpad and the seams show again.

Tokens are also the unit of accounting. Every call reports how many went in and out:

In [ ]:
# Every call reports its token counts.
response = llm.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain in two sentences what a cantilever beam is."}],
)
print(response["choices"][0]["message"]["content"])
print()
print(f"input tokens:  {response['usage']['prompt_tokens']}")
print(f"output tokens: {response['usage']['completion_tokens']}")

Why token counts matter:

- **Context window**: every model has a fixed maximum number of tokens it can handle in one call (input and output combined). Long documents and long conversations hit this limit.
- **Speed**: the model generates one token at a time. An answer twice as long takes roughly twice as long.
- **Cost**: commercial providers charge per token. The Academic Cloud is free for us, but it limits how many requests each account may send, so calls are not unlimited either.

### Try it

- Ask the model to spell a long word backwards. Check the result character by character.
- Try the letter-counting question with technical terms from your field. Find one the model gets wrong.

In [ ]:
# Scratchpad


## Part 3: Where the model runs

So far "the model" was hidden behind `llm.chat()`. This is what happens:

```
 your notebook                         GWDG Academic Cloud
 ─────────────                         ───────────────────
 llm.chat(...)   →  HTTP request  →    model server
                                         runs the model on its GPUs,
                                         generates token by token
                 ←  HTTP response ←
```

The model does not run on this machine. It runs on the **GWDG Academic Cloud**, a service that GWDG (the computing centre in Göttingen) offers to German universities. Your notebook sends the conversation over the internet, the server's GPUs generate the reply, and the answer comes back. The API key you saved in the Onboarding notebook tells the server who you are. Like any online service: do not send anything confidential.

`llm.py` is our own small client for that server, plain Python you can read in a few minutes. It speaks the **OpenAI API**: the request and response format that OpenAI introduced and that many other providers and tools now accept. So what you learn here is not tied to our setup. With a different provider, only the address, the key and the model name change.

<details><summary>What is "a model", physically?</summary>

A model is essentially one giant array of numbers: the parameters learned during training. Our everyday model has about 30 billion of them, tens of gigabytes of data, stored on the server, not on this machine. "Running the model" means streaming your tokenized input through arithmetic on these numbers until probabilities for the next token come out. There is no database and no code with if-statements about France inside, only numbers.

</details>

In [ ]:
# Ask the server which models it offers.
for name in llm.models():
    print(name)

Our everyday model is `llm.CHAT_MODEL`. Reading its name, `qwen3-30b-a3b-instruct-2507`:

- **`qwen3`**: the model family and version (Qwen 3, built by Alibaba).
- **`30b`**: about 30 billion parameters in total.
- **`a3b`**: about 3 billion of them are **active** for each token. This is a **mixture-of-experts** model: the network is split into many smaller "expert" blocks, and for every token a small router picks a few of them. Only those run, so the model is much faster than a regular 30B model while keeping most of what 30 billion parameters can store.
- **`instruct`**: fine-tuned to follow instructions and hold a conversation (the fine-tuning step from Part 1). Without it, a model only continues text.
- **`2507`**: the release, July 2025.

Rule of thumb: more parameters means more capable, but also slower and more expensive to run. The other models in the list belong to other parts of the course, or simply to other users of the service.

### Try it

Run the cell below, then consider: can you trust this answer?

In [ ]:
response = llm.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "What model are you, exactly? Who built you?"}],
)
print(response["choices"][0]["message"]["content"])

A model's claim about itself is generated like any other text, from patterns in training data. The model does not inspect itself. Models regularly misidentify themselves, especially small ones. This is Part 1 again: plausible, not verified.

## Part 4: The chat call, one argument at a time

Back to the setup cell from the top, now with every piece named:

```python
response = llm.chat(
    model=MODEL,                             # which model to use
    messages=[                               # the conversation so far
        {"role": "user", "content": "..."},  # one message: who said it + the text
    ],
)
```

- **`model`**: a name from `llm.models()`. `MODEL` holds `llm.CHAT_MODEL`.
- **`messages`**: a list of messages, oldest first. The model reads all of them and produces one reply.
- **`role`**: who a message is from. `"user"` is you, `"assistant"` is the model's own earlier replies, `"system"` is standing instructions (next section).
- **`content`**: the text of the message.

In this notebook the list always has one or two entries. Real multi-turn conversations come in the later labs.

### The prompt shapes the answer

The full input text is called the **prompt**. Since the model continues your text, the prompt is the single biggest lever you have. Run the next two cells and compare what a vague and a precise request return.

In [ ]:
# A vague prompt gets a generic answer.
response = llm.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Tell me about beams."}],
)
print(response["choices"][0]["message"]["content"])

In [ ]:
# The same topic, asked precisely.
response = llm.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "In three short bullet points, explain the difference between a cantilever beam and a simply supported beam for a first-semester mechanical engineering student."}],
)
print(response["choices"][0]["message"]["content"])

### The system message

A message with `role: "system"` carries standing instructions: who the model should be, what rules to follow, what format to answer in. It is read before the user messages and applies to the entire conversation. Products like ChatGPT put a long hidden system prompt in front of everything you type.

In [ ]:
# A system message sets standing instructions for the whole conversation.
response = llm.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a materials science tutor. Answer in one short sentence."},
        {"role": "user", "content": "Why does steel rust?"},
    ],
)
print(response["choices"][0]["message"]["content"])

### Reading the response

`llm.chat()` returns the server's answer as a Python dictionary in the OpenAI format, not just text. The reply itself lives in `response["choices"][0]["message"]`, which has the same shape as the messages you send: a `role` and a `content`. Next to it, `response["usage"]` holds the token counts from Part 2.

Some models, called **reasoning models**, also write working notes before committing to the final answer. Our everyday model is not a reasoning model, so its message has no `reasoning` field. Reasoning models (`llm.REASONING_MODEL`) put their notes in `message["reasoning"]`. Lab 05 is about reasoning models.

In [ ]:
# The response is a dictionary; the reply is one message inside it.
message = response["choices"][0]["message"]
print("role:     ", message["role"])
print("content:  ", message["content"])
print("reasoning:", message.get("reasoning"))   # None: this model does not think first
print("usage:    ", response["usage"])

### Try it

Set a system message with a rule, for example: "Always answer in German." Then try to override it from the user message: "Answer in English: why does steel rust?"

Which instruction wins? Run it a few times. There is no guaranteed winner: both messages are just text the model weighs against each other. Keep this in mind: it becomes a security topic (prompt injection) later in the course.

In [ ]:
# Scratchpad


## Part 5: Same prompt, different answers

Rerunning a cell gives a different answer. This part explains why.

The model never outputs a single next token. It assigns a probability to **every** token in its vocabulary, and the server **samples** one from that distribution:

```
prompt: "Suggest one creative name for a robot that inspects bridges."

next-token probabilities:   "Bridge..."  34%    "Span..."  21%    "Iron..."  9%   ...   (illustrative)
                                   ↓
                       sample one, append it, repeat
```

The **temperature** setting controls the sampling. At temperature 0 the model always takes the most likely token (called **greedy** decoding). Higher values flatten the distribution: less likely tokens get picked more often and the output gets more varied. Run the same prompt three times:

In [ ]:
# Same prompt, three runs, default temperature.
for i in range(3):
    response = llm.chat(
        model=MODEL,
        messages=[{"role": "user", "content": "Suggest one creative name for a robot that inspects bridges. Reply with the name only."}],
    )
    print(f"run {i + 1}: {response['choices'][0]['message']['content'].strip()}")

Now the same three runs with `temperature=0`. The model always picks the most likely token, so the runs should come out identical. If they still differ occasionally, that is the shared server: requests from several users are batched on the GPU, and rounding differences can flip a close call. Temperature 0 removes the sampling randomness, not every source of variation.

Note what temperature 0 does **not** mean: deterministic is not the same as correct. A wrong answer at temperature 0 is simply reproduced every time.

In [ ]:
# Same prompt, three runs, temperature 0.
for i in range(3):
    response = llm.chat(
        model=MODEL,
        messages=[{"role": "user", "content": "Suggest one creative name for a robot that inspects bridges. Reply with the name only."}],
        temperature=0,
    )
    print(f"run {i + 1}: {response['choices'][0]['message']['content'].strip()}")

Why this matters for the rest of the course: an **agent** calls an LLM in a loop and acts on the answers. Code that only works when the model answers exactly as expected will break. Everything we build in the next exercises has to tolerate this variation.

### Try it

- Raise the temperature: pass `temperature=1.5` and run the name prompt a few times. Where does creative tip over into unusable?
- Does temperature 0 also make the *factual* questions from Part 1 reproducible? Does it make them more correct?

In [ ]:
# Scratchpad


## Summary and where to next

What you have seen:

- An LLM is a text-continuation machine: plausible text out, no fact-checking, no memory between calls.
- The model reads tokens, not letters. Tokens are the unit of context limits, speed, and cost.
- The model runs on the GWDG Academic Cloud; `llm.py` is a small client that sends your messages there in the OpenAI format.
- `llm.chat()` takes a model name and a list of `{role, content}` messages; the reply comes back in `response["choices"][0]["message"]`.
- Answers vary because the model samples from probabilities; temperature controls how much.

In **Lab 01** the course starts on agents: what makes a program an agent, a first agent built from hand-written rules, and the same agent with its rules replaced by a single LLM call.

### Optional further material

Two free resources for the deeper story behind Part 1:

- Andrej Karpathy, [Intro to Large Language Models](https://www.youtube.com/watch?v=zjkBMFhNj_g) (1 h): training, fine-tuning, and the "LLM as OS" picture.
- 3Blue1Brown, [But what is a GPT?](https://www.youtube.com/watch?v=wjZofJX0v4M) (27 min): visual intuition for what happens inside the network.